# E3.5 · The metrics that matter at your level

**Function E — AI Governance for Agentic Systems → Running the Programme — the CISO Office**  ·  *Security of AI*

Builds on **[E3.4 · Org design and ownership](https://spbreed.github.io/cyber-commons/lessons/E3.4.html)**.

| | |
|---|---|
| Tools used | OpenSearch |

## What this lesson is

**What it covers.** Instrument the six metrics from your lab stack.

**Why a security engineer needs it.** Reporting activity instead of exposure. The control it builds is: inventory coverage, attested-identity share, standing-access reduction, MTT-revoke, blast-radius distribution, eval-gate pass rate.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Metrics that count activity — agents reviewed, policies written — demonstrate effort. Metrics that demonstrate control are about coverage, containment, verification and time-to-stop, and they are much less comfortable.

> **At CyberTravels.** How many of CyberTravels' agents are in the register, how many have egress control, and what is the median time to stop one. Not how many policies were written.

## 2 · The framework

```
   activity metrics             control metrics
   +--------------------+       +---------------------------+
   | agents reviewed    |       | % of agents in the register|
   | policies written   |  vs   | % with egress control      |
   | training completed |       | median time-to-stop        |
   +--------------------+       +---------------------------+
        comfortable                  uncomfortable, and true
```

The metrics that matter at this level are few, and none of them is a count of
alerts, findings or trainings completed.

Five numbers, each with a property that makes it worth reporting monthly:

- **Exposure** — fleet blast radius. Moves when someone adds a tool.
- **Likelihood** — red-team attack success rate. Measured, not assessed.
- **Assurance** — controls *currently* evidenced. Degrades on its own.
- **Coverage** — agents in the inventory, honestly stated as a fraction of an
  estimate.
- **Speed** — measured time-to-stop, from a game day.

The shared property is the important one: **each degrades if nobody does
anything.** A metric that stays flat under neglect is measuring activity, not
posture — which is why finding counts and training completion make such
comfortable and useless board slides.

## 3 · The procedure, as a skill

The skill applies the neglect test: simulate a quarter with nothing done and see which numbers move. Four comfortable metrics stay green throughout, which is what disqualifies them.

In [ ]:
# skills/programme/programme-metrics-selection/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: programme-metrics-selection
description: >-
  Pick programme metrics that degrade when the programme is neglected, and
  separate them from the comfortable ones that stay green regardless. Use when
  choosing what to report, or when every metric is green and the estate is not.
allowed-tools: Read, Grep, Glob
---

# A metric that cannot go red is a decoration

The test for a programme metric is simple and rarely applied: if the programme
were neglected for a quarter, would this number move? Counts of policies
published, training completed and tools deployed would not. Exposure,
containment effectiveness, evidence freshness and measured time-to-stop all
would.

## When to use this

Choosing what to report to an executive committee, and auditing an existing
dashboard that is entirely green.

## Procedure

**1 — List the candidate metrics** and, for each, what it is computed from.
Anything computed from a plan rather than from the estate is a decoration
already.

**2 — Apply the neglect test.** Simulate a quarter with nothing done: no
attestations refreshed, no evals run, no drift reviewed. Which numbers move?

**3 — Keep the ones that move and say what each one costs to compute.** A
metric nobody can produce monthly will be produced annually and stop being a
control.

**4 — Report the comfortable ones you are dropping,** with the reason. They have
constituencies, and removing them silently gets them reinstated.

**5 — Set a target and a direction per surviving metric.** A number with no
target is a chart; a number with a target is a commitment somebody is
accountable for.

## Output contract

```json
{
  "candidates": [{"name": "str", "computed_from": "estate|plan", "moves_under_neglect": false}],
  "kept": [{"name": "str", "value": 0.0, "target": 0.0, "cadence": "str", "cost": "str"}],
  "dropped": [{"name": "str", "why": "str"}],
  "neglect_simulation": {"quarter": "str", "moved": ["str"], "unmoved": ["str"]}
}
```

## Failure modes

- **Metrics computed from the plan.** They measure the plan.
- **Dropping comfortable metrics silently.** They come back.
- **No target.** Nobody is accountable for a direction.
"""

In [ ]:
# Execute the skill above, using the shared runtime rather than a copy.
import glob, importlib.util, os, sys

# Kaggle mounts an attached kernel under /kaggle/input, and it uses two
# different layouts — /kaggle/input/<slug>/ on some kernels and
# /kaggle/input/notebooks/<user>/<slug>/ on others. Both were observed on the
# same account in the same hour, so match either. The recursive glob is cheap
# here because /kaggle/input holds only what is attached; globbing the working
# tree instead cost eleven seconds a notebook.
_WHERE = (sorted(glob.glob("/kaggle/input/**/cyber-commons-skill-runtime/__script__.py",
                           recursive=True))
          + [os.path.join(p, "skills/_runtime/cyber_commons_skill_runtime.py")
             for p in (".", "..", "../..")])
_found = next((p for p in _WHERE if os.path.isfile(p)), None)
if _found is None:
    # Say what was looked for and what is actually there. "The runtime is
    # missing" on its own costs whoever hits it an afternoon.
    raise SystemExit("The shared skill runtime is missing."
                     "  looked at: " + repr(_WHERE) +
                     "  /kaggle/input holds: " +
                     repr(glob.glob("/kaggle/input/**", recursive=True)[:20]) +
                     "  cwd: " + os.getcwd() +
                     ". On Kaggle it is attached to this notebook as a "
                     "source; locally it is skills/_runtime/ in the repository.")
_spec = importlib.util.spec_from_file_location("cyber_commons_skill_runtime", _found)
cyber_commons_skill_runtime = importlib.util.module_from_spec(_spec)
sys.modules["cyber_commons_skill_runtime"] = cyber_commons_skill_runtime
_spec.loader.exec_module(cyber_commons_skill_runtime)
from cyber_commons_skill_runtime import run_skill

# Split skills/programme/programme-metrics-selection/SKILL.md into the two halves an agent uses —
# the frontmatter it routes on, and the body it follows.
meta, body = run_skill(SKILL_MD)

In [ ]:
# skills/programme/programme-metrics-selection/scripts/programme_metrics_selection.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Compute the metrics that degrade under neglect and separate them from the comfortable ones that do not.

This is the executable half of the `programme-metrics-selection` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

import time
SCOPE_WEIGHT = {"self":1,"project":3,"tenant":8,"org":20}
now = time.time(); DAY = 86400

FLEET = {"pr-remediation": [("read_file","self",True),("write_file","project",True),
                            ("deploy","org",False)],
         "claims-triage":  [("read_file","self",True),("issue_refund","tenant",False)],
         "doc-summariser": [("read_file","self",True)]}
GATED = {"pr-remediation": set(), "claims-triage": {"issue_refund"},
         "doc-summariser": set()}
def blast(t, g): return sum(SCOPE_WEIGHT[s]*(1 if rev else 2) for n,s,rev in t if n not in g)
exposure = sum(blast(t, GATED[a]) for a, t in FLEET.items())

ATTACKS = [("metadata",False),("traversal",False),("unlisted egress",True),("denied tool",False)]
asr = sum(1 for _, t in ATTACKS if t)/len(ATTACKS)

CONTROL_TESTS = {"AC-1": now-3*DAY, "AC-2": now-9*DAY, "SB-1": now-45*DAY,
                 "EV-1": now-5*DAY, "EV-2": now-12*DAY}
WINDOW = {"AC-1":30,"AC-2":30,"SB-1":30,"EV-1":60,"EV-2":30}
REQUIRED = ["AC-1","AC-2","SB-1","SB-2","EV-1","EV-2","DR-1","ST-1"]
evidenced = sum(1 for c in REQUIRED
                if c in CONTROL_TESTS and (now-CONTROL_TESTS[c])/DAY <= WINDOW[c])
assurance = evidenced/len(REQUIRED)

REGISTERED, ESTIMATED = 41, 120
coverage = REGISTERED/ESTIMATED
TIME_TO_STOP = 12

METRICS = {
 "exposure   fleet blast radius":        (exposure, "units of unreviewed action"),
 "likelihood red-team ASR":              (f"{asr:.0%}", "measured, containment surface"),
 "assurance  controls evidenced":        (f"{assurance:.0%}", f"{evidenced}/{len(REQUIRED)}"),
 "coverage   agents in inventory":       (f"{coverage:.0%}", f"{REGISTERED} of ~{ESTIMATED} est."),
 "speed      measured time-to-stop":     (f"{TIME_TO_STOP}s", "game day 41 days ago"),
}
for k, (v, note) in METRICS.items():
    print(f"{k:36s}{str(v):>8}   {note}")

COMFORTABLE = {
 "findings closed this quarter": "goes up with activity; says nothing about posture",
 "training completion %":        "reaches 98% and stays there forever",
 "number of AI policies":        "monotonically increasing by construction",
 "tools evaluated":              "measures procurement, not risk",
}
print("metrics that look like governance and are not:")
for m, why in COMFORTABLE.items():
    print(f"   {m:34s}{why}")

def degrades_under_neglect(metric):
    DEGRADES = {"exposure": True, "likelihood": True, "assurance": True,
                "coverage": True, "speed": True,
                "findings closed": False, "training completion": False,
                "number of policies": False, "tools evaluated": False}
    return DEGRADES.get(metric, False)

print(f"\n{'metric':28s}{'degrades if ignored?':>22}")
print("-" * 52)
for m in ("exposure","likelihood","assurance","coverage","speed",
          "findings closed","training completion","number of policies"):
    print(f"{m:28s}{str(degrades_under_neglect(m)):>22}")
print("\nThe first five fall on their own. That is what makes them worth")
print("reporting monthly — the report itself creates the pressure.")

def project(months, exposure, asr, assurance, coverage, ttl_days=41):
    """What happens to each metric if nobody does anything for N months."""
    new_agents_per_month = 3
    exposure_growth = 8            # blast units per new agent, ungoverned
    controls_going_stale = 0.12    # fraction of evidence expiring per month
    return {
      "exposure":  exposure + months * new_agents_per_month * exposure_growth,
      "likelihood": min(asr + months * 0.03, 1.0),
      "assurance": max(assurance - months * controls_going_stale, 0.0),
      "coverage":  max(coverage - months * 0.04, 0.0),
      "days_since_stop_test": ttl_days + months * 30,
    }

print(f"{'month':>6}{'exposure':>10}{'ASR':>7}{'assurance':>11}{'coverage':>10}"
      f"{'stop test age':>15}")
print("-" * 60)
for m in (0, 3, 6, 12):
    p = project(m, exposure, asr, assurance, coverage)
    print(f"{m:>6}{p['exposure']:>10}{p['likelihood']:>7.0%}{p['assurance']:>11.0%}"
          f"{p['coverage']:>10.0%}{p['days_since_stop_test']:>15}")

p12 = project(12, exposure, asr, assurance, coverage)
print(f"\nAfter a year of no investment: exposure {exposure}→{p12['exposure']}, "
      f"assurance {assurance:.0%}→{p12['assurance']:.0%}.")
print("Nobody made a bad decision. This is the default trajectory, and the")
print("monthly report is what makes it visible before it is a board topic.")
assert p12["assurance"] < assurance and p12["exposure"] > exposure

## What you just proved

The five metrics compute to exposure 46, ASR 25%, assurance 50%, coverage 34% and a 12-second time-to-stop. Four comfortable metrics are shown not to degrade under neglect while all five real ones do. Projected twelve months forward with no investment, exposure rises from 46 to 334 and assurance falls from 50% to 0%.

## Your turn

Which of the five can you produce today without a project? Start reporting that one monthly and let the missing ones become conspicuous — that is a cheaper way to get the others funded than asking for all five at once.

---

**Next → [E3.6 · Saying no, and saying yes with conditions](https://spbreed.github.io/cyber-commons/lessons/E3.6.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E3.5.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E3.5.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*